In [ ]:
# Scenario
# “A hospital uses different employees (agents) to handle patient care in sequence.”

# ================================
# AGENT 1: Intake Agent (Planner)
# - Collects patient symptoms and history
# - Decides which steps are needed (tests, consultations, etc.)

# ================================
# AGENT 2: Diagnostic Agent
# - Orders lab tests or scans
# - Interprets results and identifies possible conditions

# ================================
# AGENT 3: Treatment Agent
# - Suggests treatment options (medication, therapy, surgery)
# - Considers patient preferences and medical guidelines

# ================================
# AGENT 4: Decision Agent
# - Reviews all inputs (history, diagnostics, treatment options)
# - Provides the final recommendation to the patient

In [1]:
# ================================
# AGENT 1: INTAKE AGENT (Planner)
# ================================
def intake_agent(patient_query):
    print("\n[Intake Agent] Creating care plan...")
    return ["diagnostics", "treatment", "decision"]


# ================================
# AGENT 2: DIAGNOSTIC AGENT
# ================================
def diagnostic_agent():
    print("\n[Diagnostic Agent] Running tests...")
    return [
        {"condition": "Bacterial Pneumonia", "severity": "Moderate"},
        {"condition": "Viral Bronchitis",    "severity": "Mild"}
    ]


# ================================
# AGENT 3: TREATMENT AGENT
# ================================
def treatment_agent():
    print("\n[Treatment Agent] Suggesting treatments...")
    return {"medication": "Amoxicillin 500mg", "rest": "7 days", "follow_up": "3 days"}


# ================================
# AGENT 4: DECISION AGENT
# ================================
def decision_agent(diagnostics, treatment):
    print("\n[Decision Agent] Making final recommendation...")

    primary = diagnostics[0]  

    if primary["severity"] == "Severe":
        return "Admit patient immediately"

    return f"Diagnosis: {primary['condition']} → Prescribe {treatment['medication']}, follow up in {treatment['follow_up']}"


# ================================
# MAIN MULTI-AGENT SYSTEM
# ================================
def hospital_multi_agent(patient_query):
    print("Patient Query:", patient_query)

    plan = intake_agent(patient_query)

    diagnostics = None
    treatment   = None

    for step in plan:
        if step == "diagnostics":
            diagnostics = diagnostic_agent()

        elif step == "treatment":
            treatment = treatment_agent()

        elif step == "decision":
            result = decision_agent(diagnostics, treatment)

    return result


# RUN
response = hospital_multi_agent("I have fever and cough since 3 days")
print("\nFinal Recommendation:", response)

Patient Query: I have fever and cough since 3 days

[Intake Agent] Creating care plan...

[Diagnostic Agent] Running tests...

[Treatment Agent] Suggesting treatments...

[Decision Agent] Making final recommendation...

Final Recommendation: Diagnosis: Bacterial Pneumonia → Prescribe Amoxicillin 500mg, follow up in 3 days


In [8]:
# With LLM - Enhanced Multi-Agent Hospital System

from openai import OpenAI
from dotenv import load_dotenv
import os
import json
import time

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)


# UTILITY
def call_llm(system_prompt, user_message):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]
    )
    return response.choices[0].message.content


# ================================
# AGENT 1: INTAKE AGENT (Planner)
# ================================
def intake_agent(patient_query):
    print("Role     : Hospital Intake Coordinator")
    print("Task     : Analyze symptoms and create a care plan")
    print(f"Input    : {patient_query}")
    print("Status   : Generating care plan...\n")

    reply = call_llm(
        system_prompt=(
            "You are a hospital intake agent. Based on the patient's symptoms, "
            "return ONLY a Python list of steps from: diagnostics, treatment, decision. "
            "Example: ['diagnostics', 'treatment', 'decision']"
        ),
        user_message=patient_query
    )

    plan = eval(reply)

    print(f"are Plan Created  : {plan}")
    print(f"Total Steps        : {len(plan)}")
    for i, step in enumerate(plan, 1):
        emoji = {"diagnostics": "🔬", "treatment": "💊", "decision": "🩺"}.get(step, "➡️")
        print(f"   Step {i}: {emoji} {step.capitalize()}")

    return plan


# ================================
# AGENT 2: DIAGNOSTIC AGENT
# ================================
def diagnostic_agent(patient_query):
    print("Diagnostic Agent : ")
    print("Role     : Medical Diagnostics Specialist")
    print("Task     : Identify possible conditions and severity")
    print(f"Input    : {patient_query}")
    print("Status   : Running diagnostic analysis...\n")

    reply = call_llm(
        system_prompt=(
            "You are a medical diagnostic agent. Based on symptoms, return ONLY a Python list "
            "of 2 dicts with keys 'condition', 'severity' (Mild/Moderate/Severe), and 'confidence_percent' (0-100). "
            "No extra text."
        ),
        user_message=patient_query
    )

    diagnostics = eval(reply)

    print("Diagnostic Results:")
    for i, d in enumerate(diagnostics, 1):
        severity_icon = {"Mild": "🟢", "Moderate": "🟡", "Severe": "🔴"}.get(d.get("severity", ""), "⚪")
        print(f"\n   Diagnosis #{i}")
        print(f"   🏷️  Condition   : {d.get('condition', 'N/A')}")
        print(f"   {severity_icon} Severity    : {d.get('severity', 'N/A')}")
        print(f"   Confidence  : {d.get('confidence_percent', 'N/A')}%")

    return diagnostics


# ================================
# AGENT 3: TREATMENT AGENT
# ================================
def treatment_agent(diagnostics):
    print("Treatment Agent : ")
    print("Role     : Medical Treatment Specialist")
    print("Task     : Recommend medication and care instructions")
    print(f"Input    : {diagnostics}")
    print("Status   : Preparing treatment plan...\n")

    reply = call_llm(
        system_prompt=(
            "You are a medical treatment agent. Based on the diagnosis, return ONLY a Python dict with keys: "
            "medication (list of dicts with name and dosage), rest (string advice), follow_up (string), "
            "duration_days (int), warnings (list of strings). No extra text."
        ),
        user_message=str(diagnostics)
    )

    treatment = eval(reply)

    print("Treatment Plan:")

    meds = treatment.get("medication", [])
    print(f"\n    Medications ({len(meds)} prescribed):")
    for med in meds:
        if isinstance(med, dict):
            print(f"      • {med.get('name', 'N/A')} — {med.get('dosage', 'N/A')}")
        else:
            print(f"      • {med}")

    print(f"\n    Rest Advice    : {treatment.get('rest', 'N/A')}")
    print(f"   Duration       : {treatment.get('duration_days', 'N/A')} days")
    print(f"   🔁Follow-Up      : {treatment.get('follow_up', 'N/A')}")

    warnings = treatment.get("warnings", [])
    if warnings:
        print(f"\n    Warnings ({len(warnings)}):")
        for w in warnings:
            print(f"      ⚡ {w}")

    return treatment


# ================================
# AGENT 4: DECISION AGENT
# ================================
def decision_agent(patient_query, diagnostics, treatment):
    print("Decision Agent")
    print("🩺 Role     : Senior Attending Physician")
    print("📋 Task     : Final review and patient recommendation")
    print("📝 Input    : Full patient context (query + diagnosis + treatment)")
    print("⏳ Status   : Reviewing all data for final decision...\n")

    result = call_llm(
        system_prompt=(
            "You are a senior doctor. Review all inputs and give a structured final recommendation. "
            "Include: overall_assessment (1 sentence), immediate_action (1 sentence), "
            "home_care_tips (2-3 bullet points as a string), red_flags (1 sentence about when to visit ER). "
            "Format as a Python dict with these 4 keys only."
        ),
        user_message=f"Patient: {patient_query}\nDiagnosis: {diagnostics}\nTreatment: {treatment}"
    )

    try:
        decision = eval(result)
        print(" Final Medical Decision:")
        print(f"\n    Assessment      : {decision.get('overall_assessment', 'N/A')}")
        print(f"    Immediate Action: {decision.get('immediate_action', 'N/A')}")
        print(f"    Home Care Tips  :\n      {decision.get('home_care_tips', 'N/A')}")
        print(f"    Red Flags       : {decision.get('red_flags', 'N/A')}")
        return decision
    except:
        # Fallback if LLM returns plain text
        print(" Final Recommendation (plain text):")
        print(f"\n   {result}")
        return result


# ================================
# MAIN MULTI-AGENT SYSTEM
# ================================
def hospital_multi_agent(patient_query):
    print("Multi Agent System : ")
    print(" System   : AI-Powered Multi-Agent Medical Assistant")
    print(" Model    : LLaMA 3.3 70B (via Groq)")
    print(f" Patient  : {patient_query}")
    print(f" Time     : {time.strftime('%Y-%m-%d %H:%M:%S')}")

    plan       = intake_agent(patient_query)
    diagnostics = None
    treatment   = None
    result      = None

    for step in plan:
        if step == "diagnostics":
            diagnostics = diagnostic_agent(patient_query)

        elif step == "treatment":
            treatment = treatment_agent(diagnostics)

        elif step == "decision":
            result = decision_agent(patient_query, diagnostics, treatment)

    print(" All agents executed successfully.")
    print(f" Steps Completed: {' → '.join([s.capitalize() for s in plan])}")

    return result


# ================================
# RUN
# ================================
response = hospital_multi_agent("I have fever and cough since 3 days")
print("\n FINAL OUTPUT:")
print(response)

Multi Agent System : 
 System   : AI-Powered Multi-Agent Medical Assistant
 Model    : LLaMA 3.3 70B (via Groq)
 Patient  : I have fever and cough since 3 days
 Time     : 2026-03-19 11:42:42
Role     : Hospital Intake Coordinator
Task     : Analyze symptoms and create a care plan
Input    : I have fever and cough since 3 days
Status   : Generating care plan...

are Plan Created  : ['diagnostics', 'treatment', 'decision']
Total Steps        : 3
   Step 1: 🔬 Diagnostics
   Step 2: 💊 Treatment
   Step 3: 🩺 Decision
Diagnostic Agent : 
Role     : Medical Diagnostics Specialist
Task     : Identify possible conditions and severity
Input    : I have fever and cough since 3 days
Status   : Running diagnostic analysis...

Diagnostic Results:

   Diagnosis #1
   🏷️  Condition   : Common Cold
   🟢 Severity    : Mild
   Confidence  : 60%

   Diagnosis #2
   🏷️  Condition   : Bronchitis
   🟡 Severity    : Moderate
   Confidence  : 40%
Treatment Agent : 
Role     : Medical Treatment Specialist
Task

In [ ]:
# Scenario: Corporate Market Research & Strategy
# A company wants to explore launching a new product in a competitive market. The Manager Agent oversees the process and delegates tasks to specialized workers.

# 👩‍💼 Manager Agent
# - Receives the overall goal: “Evaluate feasibility of launching Product Y in Asia.”
# - Dynamically assigns tasks to worker agents depending on what’s needed.
# - Example: If budget is unclear → send to Finance Worker. If regulations are complex → send to Legal Worker.

# ================================
# 📊 Worker Agents
# ================================
# - Market Research Worker
# - Collects competitor data, customer preferences, and demand forecasts.
# - Reports: “High demand in Tier‑1 cities, moderate competition.”
# - Finance Worker
# - Analyzes budget, ROI, and pricing strategy.
# - Reports: “Estimated $10M investment, ROI in 2 years.”
# - Operations Worker
# - Evaluates supply chain, production capacity, and logistics.
# - Reports: “Factories can scale up, but shipping costs are high.”
# - Legal Worker
# - Reviews compliance, intellectual property, and regional regulations.
# - Reports: “Trademark available, but import laws require certification.”
# - HR Worker
# - Assesses staffing needs and training requirements.
# - Reports: “Need 50 new hires for customer support and sales.”

In [11]:
# CORPORATE MARKET RESEARCH & STRATEGY
# Manager–Worker Multi-Agent System (No LLM)

# ================================
# WORKER AGENTS
# ================================

def market_research_worker():
    print("\n  [Market Research Worker] Collecting data...")
    return {
        "demand_level": "High",
        "target_cities": ["Shanghai", "Tokyo", "Mumbai"],
        "competition_level": "Moderate",
        "summary": "High demand in Tier-1 cities, moderate competition."
    }

def finance_worker():
    print("\n  [Finance Worker] Analyzing budget & ROI...")
    return {
        "estimated_investment_usd": 10_000_000,
        "projected_roi_years": 2,
        "risk_level": "Medium",
        "summary": "Estimated $10M investment, ROI in 2 years."
    }

def operations_worker():
    print("\n  [Operations Worker] Evaluating supply chain...")
    return {
        "factory_scalability": "High",
        "shipping_bottleneck": True,
        "bottleneck_reason": "High last-mile delivery costs",
        "summary": "Factories can scale up, but shipping costs are high."
    }

def legal_worker():
    print("\n  [Legal Worker] Reviewing regulations...")
    return {
        "trademark_available": True,
        "import_certification_required": True,
        "certification_name": "CE Mark + ASEAN Compliance",
        "summary": "Trademark available, but import laws require certification."
    }

def hr_worker():
    print("\n  [HR Worker] Assessing staffing needs...")
    return {
        "new_hires_required": 50,
        "roles": {"Customer Support": 20, "Sales": 20, "Operations": 10},
        "summary": "Need 50 new hires for customer support and sales."
    }


# ================================
# MANAGER AGENT
# ================================
def manager_agent(goal):
    print("\n[Manager Agent] Analyzing goal...")
    print(f"  Goal: {goal}\n")

    tasks = []
    g = goal.lower()

    if any(k in g for k in ["market", "demand", "competitor", "feasibility", "launch"]):
        tasks.append("market_research")
    if any(k in g for k in ["budget", "finance", "roi", "investment", "feasibility"]):
        tasks.append("finance")
    if any(k in g for k in ["supply", "operations", "logistics", "feasibility"]):
        tasks.append("operations")
    if any(k in g for k in ["legal", "compliance", "regulation", "feasibility"]):
        tasks.append("legal")
    if any(k in g for k in ["hire", "staff", "hr", "workforce", "feasibility"]):
        tasks.append("hr")

    print(f"  Tasks Assigned: {[t.replace('_', ' ').title() for t in tasks]}")
    return tasks


# ================================
# FINAL DECISION
# ================================
def make_final_decision(results):
    score, flags, positives = 0, [], []

    mr  = results.get("market_research", {})
    fin = results.get("finance", {})
    ops = results.get("operations", {})
    leg = results.get("legal", {})
    hr  = results.get("hr", {})

    if mr.get("demand_level") == "High":
        score += 1; positives.append("Strong demand in Tier-1 cities")
    if fin.get("projected_roi_years", 99) <= 3:
        score += 1; positives.append(f"ROI in {fin['projected_roi_years']} years")
    if ops.get("factory_scalability") == "High":
        score += 1; positives.append("Factories can scale up")
    if leg.get("trademark_available"):
        score += 1; positives.append("Trademark available")
    if hr.get("new_hires_required", 999) <= 100:
        score += 1; positives.append(f"{hr['new_hires_required']} hires needed — manageable")

    if ops.get("shipping_bottleneck"):   flags.append(f"Shipping: {ops['bottleneck_reason']}")
    if leg.get("import_certification_required"): flags.append(f"Certification needed: {leg['certification_name']}")
    if fin.get("risk_level") == "High":  flags.append("High financial risk")

    pct = (score / 5) * 100
    verdict = "PROCEED" if pct >= 75 else " CONDITIONAL" if pct >= 50 else " NOT RECOMMENDED"

    return {"score": f"{pct:.0f}%", "verdict": verdict, "positives": positives, "flags": flags}


# ================================
# MAIN SYSTEM
# ================================
def run_system(goal):
    print("=" * 55)
    print("  CORPORATE MARKET RESEARCH & STRATEGY SYSTEM")
    print("=" * 55)

    tasks   = manager_agent(goal)
    results = {}

    worker_map = {
        "market_research": market_research_worker,
        "finance":         finance_worker,
        "operations":      operations_worker,
        "legal":           legal_worker,
        "hr":              hr_worker
    }

    for task in tasks:
        results[task] = worker_map[task]()


    # Final Decision
    decision = make_final_decision(results)

    print("  MANAGER'S FINAL DECISION")
    print(f"\n  Feasibility : {decision['score']}  |  Verdict: {decision['verdict']}")

    print("\n  Positives:")
    for p in decision["positives"]: print(f"     • {p}")

    print("\n   Flags:")
    for f in decision["flags"]:     print(f"      {f}")
    if not decision["flags"]:       print("     None — all clear!")

    return decision


# RUN
run_system("Evaluate feasibility of launching Product Y in Asia.")

  CORPORATE MARKET RESEARCH & STRATEGY SYSTEM

[Manager Agent] Analyzing goal...
  Goal: Evaluate feasibility of launching Product Y in Asia.

  Tasks Assigned: ['Market Research', 'Finance', 'Operations', 'Legal', 'Hr']

  [Market Research Worker] Collecting data...

  [Finance Worker] Analyzing budget & ROI...

  [Operations Worker] Evaluating supply chain...

  [Legal Worker] Reviewing regulations...

  [HR Worker] Assessing staffing needs...
  MANAGER'S FINAL DECISION

  Feasibility : 100%  |  Verdict: PROCEED

  Positives:
     • Strong demand in Tier-1 cities
     • ROI in 2 years
     • Factories can scale up
     • Trademark available
     • 50 hires needed — manageable

   Flags:
      Shipping: High last-mile delivery costs
      Certification needed: CE Mark + ASEAN Compliance


{'score': '100%',
 'verdict': 'PROCEED',
 'positives': ['Strong demand in Tier-1 cities',
  'ROI in 2 years',
  'Factories can scale up',
  'Trademark available',
  '50 hires needed — manageable'],
 'flags': ['Shipping: High last-mile delivery costs',
  'Certification needed: CE Mark + ASEAN Compliance']}

In [14]:
#USing LLM


# CORPORATE MARKET RESEARCH & STRATEGY

from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

# ================================
# LLM HELPER
# ================================
def call_llm(system_prompt, user_message):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]
    )
    return response.choices[0].message.content


# ================================
# WORKER AGENTS
# ================================

def market_research_worker(goal):
    print("\n  [Market Research Worker] Collecting data...")
    reply = call_llm(
        system_prompt=(
            "You are a market research analyst. Based on the product launch goal, "
            "return ONLY a Python dict with keys: demand_level (High/Medium/Low), "
            "target_cities (list of 3), competition_level (High/Moderate/Low), summary (1 sentence). No extra text."
        ),
        user_message=goal
    )
    return eval(reply)


def finance_worker(goal):
    print("\n  [Finance Worker] Analyzing budget & ROI...")
    reply = call_llm(
        system_prompt=(
            "You are a financial analyst. Based on the product launch goal, "
            "return ONLY a Python dict with keys: estimated_investment_usd (int), "
            "projected_roi_years (int), risk_level (Low/Medium/High), summary (1 sentence). No extra text."
        ),
        user_message=goal
    )
    return eval(reply)


def operations_worker(goal):
    print("\n  [Operations Worker] Evaluating supply chain...")
    reply = call_llm(
        system_prompt=(
            "You are an operations manager. Based on the product launch goal, "
            "return ONLY a Python dict with keys: factory_scalability (High/Medium/Low), "
            "shipping_bottleneck (True/False), bottleneck_reason (string), summary (1 sentence). No extra text."
        ),
        user_message=goal
    )
    return eval(reply)


def legal_worker(goal):
    print("\n  [Legal Worker] Reviewing regulations...")
    reply = call_llm(
        system_prompt=(
            "You are a legal compliance officer. Based on the product launch goal, "
            "return ONLY a Python dict with keys: trademark_available (True/False), "
            "import_certification_required (True/False), certification_name (string), summary (1 sentence). No extra text."
        ),
        user_message=goal
    )
    return eval(reply)


def hr_worker(goal):
    print("\n  [HR Worker] Assessing staffing needs...")
    reply = call_llm(
        system_prompt=(
            "You are an HR manager. Based on the product launch goal, "
            "return ONLY a Python dict with keys: new_hires_required (int), "
            "roles (dict with role names and counts), summary (1 sentence). No extra text."
        ),
        user_message=goal
    )
    return eval(reply)


# ================================
# MANAGER AGENT
# ================================
def manager_agent(goal):
    print("\n[Manager Agent] Analyzing goal...")
    print(f"  Goal: {goal}\n")

    reply = call_llm(
        system_prompt=(
            "You are a corporate manager. Based on the business goal, decide which departments to involve. "
            "Return ONLY a Python list with any of: 'market_research', 'finance', 'operations', 'legal', 'hr'. "
            "Include all that are relevant. No extra text."
        ),
        user_message=goal
    )

    tasks = eval(reply)
    print(f"  Tasks Assigned: {[t.replace('_', ' ').title() for t in tasks]}")
    return tasks


# ================================
# FINAL DECISION
# ================================
def make_final_decision(results):
    score, flags, positives = 0, [], []

    mr  = results.get("market_research", {})
    fin = results.get("finance", {})
    ops = results.get("operations", {})
    leg = results.get("legal", {})
    hr  = results.get("hr", {})

    if mr.get("demand_level")        == "High":  score += 1; positives.append("Strong demand in Tier-1 cities")
    if fin.get("projected_roi_years", 99) <= 3:  score += 1; positives.append(f"ROI in {fin.get('projected_roi_years')} years")
    if ops.get("factory_scalability") == "High": score += 1; positives.append("Factories can scale up")
    if leg.get("trademark_available"):            score += 1; positives.append("Trademark available")
    if hr.get("new_hires_required", 999) <= 100: score += 1; positives.append(f"{hr.get('new_hires_required')} hires needed — manageable")

    if ops.get("shipping_bottleneck"):               flags.append(f"Shipping: {ops.get('bottleneck_reason')}")
    if leg.get("import_certification_required"):     flags.append(f"Certification needed: {leg.get('certification_name')}")
    if fin.get("risk_level") == "High":              flags.append("High financial risk")

    pct     = (score / 5) * 100
    verdict = "PROCEED" if pct >= 75 else "  CONDITIONAL" if pct >= 50 else " NOT RECOMMENDED"

    return {"score": f"{pct:.0f}%", "verdict": verdict, "positives": positives, "flags": flags}


# ================================
# MAIN SYSTEM
# ================================
def run_system(goal):
    print("  CORPORATE MARKET RESEARCH & STRATEGY SYSTEM")

    tasks   = manager_agent(goal)
    results = {}

    worker_map = {
        "market_research": market_research_worker,
        "finance":         finance_worker,
        "operations":      operations_worker,
        "legal":           legal_worker,
        "hr":              hr_worker
    }

    for task in tasks:
        if task in worker_map:
            results[task] = worker_map[task](goal)

    # Worker Summaries
    print("  WORKER REPORTS")
    for agent, data in results.items():
        print(f"\n   {agent.replace('_', ' ').title()}")
        print(f"     → {data['summary']}")

    # Final Decision
    decision = make_final_decision(results)

    print("  MANAGER'S FINAL DECISION")
    print(f"\n  Feasibility : {decision['score']}  |  Verdict: {decision['verdict']}")

    print("\n   Positives:")
    for p in decision["positives"]: print(f"     • {p}")

    print("\n    Flags:")
    for f in decision["flags"]:     print(f"      {f}")
    if not decision["flags"]:       print("     None — all clear!")

    return decision


# RUN
run_system("Evaluate feasibility of launching Product Y in Asia.")

  CORPORATE MARKET RESEARCH & STRATEGY SYSTEM

[Manager Agent] Analyzing goal...
  Goal: Evaluate feasibility of launching Product Y in Asia.

  Tasks Assigned: ['Market Research', 'Finance', 'Operations', 'Legal']

  [Market Research Worker] Collecting data...

  [Finance Worker] Analyzing budget & ROI...

  [Operations Worker] Evaluating supply chain...

  [Legal Worker] Reviewing regulations...
  WORKER REPORTS

   Market Research
     → Product Y has a high demand in Asian markets, particularly in major cities like Tokyo, Seoul, and Hong Kong, with moderate competition from established brands.

   Finance
     → Launching Product Y in Asia is expected to yield moderate returns within a reasonable timeframe, considering the region's growing market demand.

   Operations
     → Launching Product Y in Asia is feasible but will require strategic planning to address shipping logistics and warehouse capacity limitations.

   Legal
     → Product Y can be launched in Asia after obtaining 

{'score': '60%',
 'verdict': '  CONDITIONAL',
 'positives': ['Strong demand in Tier-1 cities',
  'ROI in 3 years',
  'Trademark available'],
 'flags': ['Shipping: insufficient warehouse space in target region',
  'Certification needed: CCC Certification']}

In [ ]:
# Agent 1: Crisis Coordinator (Broadcaster)
# - Broadcasts: “Data breach detected in customer database. Immediate response required.”
# - Sends this to all other agents simultaneously.

# 🛡️ Agent 2: IT Security Agent
# - Receives broadcast.
# - Responds: “Isolate affected servers, patch vulnerabilities, start forensic analysis.”

# 📞 Agent 3: Communications Agent
# - Receives broadcast.
# - Responds: “Draft internal memo, prepare press release, notify stakeholders.”

# 💰 Agent 4: Finance Agent
# - Receives broadcast.
# - Responds: “Estimate financial impact, allocate emergency funds, review insurance coverage.”

# 👩‍⚖️ Agent 5: Legal Agent
# - Receives broadcast.
# - Responds: “Assess regulatory obligations, prepare compliance reports, advise on liability.”

# 👩‍💼 Agent 6: HR Agent
# - Receives broadcast.
# - Responds: “Brief employees, provide guidance on handling customer queries, ensure morale support.”

# 🧑‍⚖️ Agent 7: Decision Agent (Coordinator)
# - Collects all responses.
# - Integrates into a final crisis response plan:
# “Servers isolated, communications prepared, financial impact assessed, compliance secured, employees briefed.”

In [15]:
# CORPORATE CRISIS MANAGEMENT SYSTEM

# ================================
# AGENT 2-6: WORKER AGENTS
# ================================

def it_security_agent(broadcast):
    print("\n  [IT Security Agent] Received broadcast. Responding...")
    return {
        "agent": "IT Security",
        "actions": [
            "Isolate affected servers",
            "Patch vulnerabilities",
            "Start forensic analysis"
        ],
        "status": "Servers isolated and forensic analysis initiated."
    }

def communications_agent(broadcast):
    print("\n  [Communications Agent] Received broadcast. Responding...")
    return {
        "agent": "Communications",
        "actions": [
            "Draft internal memo",
            "Prepare press release",
            "Notify stakeholders"
        ],
        "status": "Communications prepared and stakeholders notified."
    }

def finance_agent(broadcast):
    print("\n  [Finance Agent] Received broadcast. Responding...")
    return {
        "agent": "Finance",
        "actions": [
            "Estimate financial impact",
            "Allocate emergency funds",
            "Review insurance coverage"
        ],
        "status": "Financial impact assessed and emergency funds allocated."
    }

def legal_agent(broadcast):
    print("\n  [Legal Agent] Received broadcast. Responding...")
    return {
        "agent": "Legal",
        "actions": [
            "Assess regulatory obligations",
            "Prepare compliance reports",
            "Advise on liability"
        ],
        "status": "Compliance secured and liability assessed."
    }

def hr_agent(broadcast):
    print("\n  [HR Agent] Received broadcast. Responding...")
    return {
        "agent": "HR",
        "actions": [
            "Brief all employees",
            "Provide guidance on customer queries",
            "Ensure morale support"
        ],
        "status": "Employees briefed and morale support initiated."
    }


# ================================
# AGENT 1: CRISIS COORDINATOR
# ================================
def crisis_coordinator():
    broadcast = "Data breach detected in customer database. Immediate response required."
    print("\n[Crisis Coordinator] Broadcasting emergency alert...")
    print(f"   Broadcast: \"{broadcast}\"")
    print("   Sending to: IT Security, Communications, Finance, Legal, HR")
    return broadcast


# ================================
# AGENT 7: DECISION AGENT
# ================================
def decision_agent(responses):
    print("\n[Decision Agent] Collecting all responses...")

    final_plan = " | ".join([r["status"] for r in responses])

    return {
        "total_agents_responded": len(responses),
        "final_crisis_plan": final_plan,
        "verdict": "Crisis response plan activated across all departments."
    }


# ================================
# MAIN SYSTEM
# ================================
def run_crisis_system():
    print("   CORPORATE CRISIS MANAGEMENT SYSTEM")

    # Agent 1: Broadcast
    broadcast = crisis_coordinator()

    # Agents 2–6: All receive broadcast simultaneously
    agent_map = {
        "IT Security":    it_security_agent,
        "Communications": communications_agent,
        "Finance":        finance_agent,
        "Legal":          legal_agent,
        "HR":             hr_agent
    }

    responses = []
    for name, agent in agent_map.items():
        responses.append(agent(broadcast))

    # Individual Reports
    print("  AGENT RESPONSES")
    for r in responses:
        print(f"\n   {r['agent']} Agent")
        for action in r["actions"]:
            print(f"     • {action}")

    # Agent 7: Final Decision
    decision = decision_agent(responses)

    print("  FINAL CRISIS RESPONSE PLAN")
    print(f"\n  Agents Responded : {decision['total_agents_responded']}")
    print(f"\n  Plan: {decision['final_crisis_plan']}")
    print(f"\n  {decision['verdict']}")

    return decision


# RUN
run_crisis_system()

   CORPORATE CRISIS MANAGEMENT SYSTEM

[Crisis Coordinator] Broadcasting emergency alert...
   Broadcast: "Data breach detected in customer database. Immediate response required."
   Sending to: IT Security, Communications, Finance, Legal, HR

  [IT Security Agent] Received broadcast. Responding...

  [Communications Agent] Received broadcast. Responding...

  [Finance Agent] Received broadcast. Responding...

  [Legal Agent] Received broadcast. Responding...

  [HR Agent] Received broadcast. Responding...
  AGENT RESPONSES

   IT Security Agent
     • Isolate affected servers
     • Patch vulnerabilities
     • Start forensic analysis

   Communications Agent
     • Draft internal memo
     • Prepare press release
     • Notify stakeholders

   Finance Agent
     • Estimate financial impact
     • Allocate emergency funds
     • Review insurance coverage

   Legal Agent
     • Assess regulatory obligations
     • Prepare compliance reports
     • Advise on liability

   HR Agent
     

{'total_agents_responded': 5,
 'final_crisis_plan': 'Servers isolated and forensic analysis initiated. | Communications prepared and stakeholders notified. | Financial impact assessed and emergency funds allocated. | Compliance secured and liability assessed. | Employees briefed and morale support initiated.',
 'verdict': 'Crisis response plan activated across all departments.'}

In [ ]:
# Scenario: Corporate Product Launch Broadcast
# Imagine a company preparing to launch Product X in Q3. The Coordinator Agent (like a corporate program manager) sends out a broadcast announcement to all departments at once:
# “Product X launch in Q3, target market North America, budget $5M.”


# 📢 Coordinator Agent (Broadcaster)
# - Sends the launch announcement to all departments simultaneously.
# - This is implemented in the code by the broadcast() function, which uses asyncio.gather() to run all agents in parallel.

# 📈 Marketing Agent
# - Receives the broadcast.
# - Responds with a marketing strategy: campaigns, channels, and positioning.
# 💰 Finance Agent
# - Receives the broadcast.
# - Responds with budget allocation and ROI forecasts.
# 🏭 Operations Agent
# - Receives the broadcast.
# - Responds with production and supply chain actions.
# 👩‍⚖️ Legal Agent
# - Receives the broadcast.
# - Responds with compliance checks and contract actions.
# 👩‍💼 HR Agent
# - Receives the broadcast.
# - Responds with staffing and training plans.

# 🧑‍⚖️ Decision Agent
# - Collects all responses.
# - Integrates them into a Final Corporate Launch Plan.
# - In the code, this is the decision_agent() function that merges all outputs into one consolidated plan.

In [17]:
# CORPORATE PRODUCT LAUNCH BROADCAST SYSTEM

import asyncio

# ================================
# WORKER AGENTS (Async)
# ================================

async def marketing_agent(broadcast):
    print("\n  [Marketing Agent] Received broadcast. Responding...")
    await asyncio.sleep(1) 
    return {
        "agent": "Marketing",
        "actions": [
            "Launch digital campaigns on LinkedIn & Instagram",
            "Partner with North America influencers",
            "Position Product X as premium B2B solution"
        ],
        "status": "Marketing campaigns planned and channels confirmed."
    }

async def finance_agent(broadcast):
    print("\n  [Finance Agent] Received broadcast. Responding...")
    await asyncio.sleep(1)
    return {
        "agent": "Finance",
        "actions": [
            "Allocate $2M for marketing, $1.5M for ops, $1M for legal & HR",
            "Forecast ROI within 18 months",
            "Set up Q3 budget tracking dashboard"
        ],
        "status": "Budget allocated and ROI forecast prepared."
    }

async def operations_agent(broadcast):
    print("\n  [Operations Agent] Received broadcast. Responding...")
    await asyncio.sleep(1)
    return {
        "agent": "Operations",
        "actions": [
            "Scale production capacity by 40%",
            "Finalize North America distribution partners",
            "Set up inventory buffer for Q3 demand"
        ],
        "status": "Production scaled and supply chain finalized."
    }

async def legal_agent(broadcast):
    print("\n  [Legal Agent] Received broadcast. Responding...")
    await asyncio.sleep(1)
    return {
        "agent": "Legal",
        "actions": [
            "Register Product X trademark in North America",
            "Review distributor contracts and NDAs",
            "Ensure FTC compliance for marketing materials"
        ],
        "status": "Trademark filed and compliance checks completed."
    }

async def hr_agent(broadcast):
    print("\n  [HR Agent] Received broadcast. Responding...")
    await asyncio.sleep(1)
    return {
        "agent": "HR",
        "actions": [
            "Hire 30 sales reps for North America region",
            "Schedule product training for all teams",
            "Launch internal awareness campaign"
        ],
        "status": "Staffing plan ready and training scheduled."
    }


# ================================
# COORDINATOR AGENT (Broadcaster)
# ================================
def coordinator_agent():
    broadcast = "Product X launch in Q3, target market North America, budget $5M."
    print("\n[Coordinator Agent] Broadcasting launch announcement...")
    print(f" Broadcast : \"{broadcast}\"")
    print("  Sending to : Marketing, Finance, Operations, Legal, HR")
    return broadcast


# ================================
# BROADCAST (Async Parallel Execution)
# ================================
async def broadcast(broadcast_message):
    print("\n[System] All agents activated simultaneously via asyncio.gather()...")

    responses = await asyncio.gather(
        marketing_agent(broadcast_message),
        finance_agent(broadcast_message),
        operations_agent(broadcast_message),
        legal_agent(broadcast_message),
        hr_agent(broadcast_message)
    )

    return list(responses)


# ================================
# DECISION AGENT
# ================================
def decision_agent(responses):
    print("\n[Decision Agent] Integrating all department responses...")

    final_plan = " | ".join([r["status"] for r in responses])

    return {
        "total_agents_responded": len(responses),
        "final_launch_plan": final_plan,
        "verdict": "Product X Corporate Launch Plan activated across all departments."
    }


# ================================
# MAIN SYSTEM
# ================================
async def run_launch_system():

    print("   CORPORATE PRODUCT LAUNCH BROADCAST SYSTEM")

    # Step 1: Coordinator broadcasts
    message = coordinator_agent()

    # Step 2: All agents respond in parallel
    responses = await broadcast(message)

    # Step 3: Individual reports
    print("  DEPARTMENT RESPONSES")
    for r in responses:
        print(f"\n   {r['agent']} Agent")
        for action in r["actions"]:
            print(f"     • {action}")

    # Step 4: Decision agent consolidates
    decision = decision_agent(responses)

    print("  FINAL CORPORATE LAUNCH PLAN")
    print(f"\n  Departments Responded : {decision['total_agents_responded']}")
    print(f"\n   Plan : {decision['final_launch_plan']}")
    print(f"\n  {decision['verdict']}")

    return decision


# RUN
await run_launch_system()

   CORPORATE PRODUCT LAUNCH BROADCAST SYSTEM

[Coordinator Agent] Broadcasting launch announcement...
 Broadcast : "Product X launch in Q3, target market North America, budget $5M."
  Sending to : Marketing, Finance, Operations, Legal, HR

[System] All agents activated simultaneously via asyncio.gather()...

  [Marketing Agent] Received broadcast. Responding...

  [Finance Agent] Received broadcast. Responding...

  [Operations Agent] Received broadcast. Responding...

  [Legal Agent] Received broadcast. Responding...

  [HR Agent] Received broadcast. Responding...
  DEPARTMENT RESPONSES

   Marketing Agent
     • Launch digital campaigns on LinkedIn & Instagram
     • Partner with North America influencers
     • Position Product X as premium B2B solution

   Finance Agent
     • Allocate $2M for marketing, $1.5M for ops, $1M for legal & HR
     • Forecast ROI within 18 months
     • Set up Q3 budget tracking dashboard

   Operations Agent
     • Scale production capacity by 40%
     

{'total_agents_responded': 5,
 'final_launch_plan': 'Marketing campaigns planned and channels confirmed. | Budget allocated and ROI forecast prepared. | Production scaled and supply chain finalized. | Trademark filed and compliance checks completed. | Staffing plan ready and training scheduled.',
 'verdict': 'Product X Corporate Launch Plan activated across all departments.'}